# Os lances de cada partida

## Demanda da aula 4

Até aqui, todas as estatísticas da Arena Analytics saíram das partidas: placar, público, fase. Mas, desde o começo do
projeto, a pasta `data/events` guarda outro conjunto de dados: o **feed de eventos**, com uma linha para cada **lance**
de cada partida (gol, cartão, substituição e, nas partidas com o feed completo, também passe, chute, disputa de bola e
falta), com o minuto e a seleção.

O time de analytics pediu um número que só os lances têm: **quantos passes cada seleção trocou nas
Copas de 2018 e 2022**, e quantos por partida, em média. São as duas Copas em que o feed traz todos os passes de todas
as partidas. A entrega são os passes gravados em
Parquet, por partida e por Copa.

O feed é o maior volume do projeto. Por isso, a cada passo da entrega, também vamos analisar **como o Spark executa**:
o que ele lê do arquivo, quando precisa trocar dados entre partições e como junta uma tabela grande com uma pequena.

| Etapa | O que fazer | Notebook |
|---|---|---|
| Task 1 | Conferir se os gols do feed batem com os placares das partidas | `01_explain` |
| Task 2 | Trazer o ano da Copa para cada lance | `02_broadcast_join` |
| Task 3 | Calcular os passes de cada seleção nas Copas de 2018 e 2022 e a média por partida | `03_shuffle_e_aqe` |
| Task 4 | Gravar os passes por partida e por Copa em Parquet e conferir o job na Spark UI | `04_parquet_e_spark_ui` |

**Pronto quando:** os gols do feed estão conferidos com os placares, e os passes de cada seleção em 2018 e 2022 estão
gravados em Parquet, por partida e por Copa, com as partidas e a média de passes por partida.

## Task 1: conferir se os gols do feed batem com os placares das partidas

In [1]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-lances")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS = "../../data/events"

spark

## O feed de eventos

O feed está em **Parquet**, um formato que guarda o schema junto com o dado.

In [2]:
eventos = spark.read.parquet(EVENTOS).withColumnRenamed("MatchID", "match_id")
eventos.show(5)

+--------+--------+------+------+----------+------+---------+
|event_id|match_id|period|minute|event_type|  team|player_id|
+--------+--------+------+------+----------+------+---------+
|       1|       1|    1H|    19|      Goal|France|     5470|
|       2|       1|    1H|    40|      Goal|France|    99087|
|       3|       1|    1H|    43|      Goal|France|    60620|
|       4|       1|    2H|    70|      Goal|Mexico|    94135|
|       5|       1|    2H|    87|      Goal|France|    60620|
+--------+--------+------+------+----------+------+---------+
only showing top 5 rows


In [3]:
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)

partidas.show(5)

+----+--------+-------+-------------+-------------+--------------+---------+-------+-------------------+----------+---------+
| ano|match_id|   fase|     mandante|gols_mandante|gols_visitante|visitante|publico|       data_partida|total_gols|resultado|
+----+--------+-------+-------------+-------------+--------------+---------+-------+-------------------+----------+---------+
|1930|       1|Group 1|       France|            4|             1|   Mexico|   4444|1930-07-13 15:00:00|         5| mandante|
|1930|       2|Group 4|United States|            3|             0|  Belgium|  18346|1930-07-13 15:00:00|         3| mandante|
|1930|       3|Group 2|   Yugoslavia|            2|             1|   Brazil|  24059|1930-07-14 12:45:00|         3| mandante|
|1930|       4|Group 3|      Romania|            3|             1|     Peru|   2549|1930-07-14 14:50:00|         4| mandante|
|1930|       5|Group 1|    Argentina|            1|             0|   France|  23409|1930-07-15 16:00:00|         1| ma

In [5]:
print("lances:", eventos.count())
print("partidas:", partidas.count())

lances: 195747
partidas: 964


Uma tabela **grande** (os lances) e uma **pequena** (as partidas).

## Os gols batem com os placares?

Se o feed estiver completo, o número de lances de gol é igual à soma de `total_gols` das partidas.

In [11]:
gols_no_feed = (
    eventos
    .filter(F.col("event_type") == "Goal")
    .select("match_id", "minute", "team")
)
gols_no_feed.show(5)

print("gols no feed:", gols_no_feed.count())

+--------+------+------+
|match_id|minute|  team|
+--------+------+------+
|       1|    19|France|
|       1|    40|France|
|       1|    43|France|
|       1|    70|Mexico|
|       1|    87|France|
+--------+------+------+
only showing top 5 rows
gols no feed: 2720


In [12]:
partidas.select(F.sum("total_gols")).first()[0]

2720

## O que o Spark leu do arquivo

Quando a ação chega, o Spark enxerga o pedido inteiro (lazy evaluation, aula 2), o otimizador (**Catalyst**) reescreve
o plano, e o **plano físico** decide como executar. O `explain()` mostra o plano físico.

In [13]:
gols_no_feed.explain(mode="formatted")

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [MatchID#1, minute#3, event_type#4, team#5]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/data/events]
PushedFilters: [IsNotNull(event_type), EqualTo(event_type,Goal)]
ReadSchema: struct<MatchID:int,minute:int,event_type:string,team:string>

(2) ColumnarToRow [codegen id : 1]
Input [4]: [MatchID#1, minute#3, event_type#4, team#5]

(3) Filter [codegen id : 1]
Input [4]: [MatchID#1, minute#3, event_type#4, team#5]
Condition : (isnotnull(event_type#4) AND (event_type#4 = Goal))

(4) Project [codegen id : 1]
Output [3]: [MatchID#1 AS match_id#8, minute#3, team#5]
Input [4]: [MatchID#1, minute#3, event_type#4, team#5]




A árvore se lê **de baixo para cima**.

| No código | No plano |
|---|---|
| `spark.read.parquet(EVENTOS)` | `Scan parquet` |
| (nenhuma linha: o Spark coloca sozinho) | `ColumnarToRow` |
| `.filter(F.col("event_type") == "Goal")` | `Filter` |
| `.select("match_id", "minute", "team")` | `Project` |

Alguns detalhes`:

- **`ReadSchema`** (Scan): as colunas que o Spark vai realmente ler (**column pruning**).
- **`PushedFilters`** (Scan): os filtros levados para a leitura (**predicate pushdown**).
- **`ColumnarToRow`**: o Parquet guarda os dados por coluna, e o Spark lê o arquivo em lotes de colunas, que é o Batched: true no Scan parquet. O Filter e o Project trabalham linha a linha, então o Spark converte os lotes em linhas antes de passar adiante.

## Para consultar depois

- **Modos do `explain`:** `explain()` mostra o plano físico; `explain(True)` mostra também os planos lógicos, antes e
  depois do otimizador; `explain(mode="formatted")` separa a árvore dos detalhes.
- **A ordem não importa:** filtrar antes ou depois do `select` gera o mesmo plano, porque o otimizador reescreve a
  consulta.
- **Em SQL:** `spark.sql("EXPLAIN SELECT ...")` mostra o plano de uma consulta SQL.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [14]:
spark.stop()